# Example 1.1.2 / 3: Giving Members in Data, and Changing Them with let

Companion notebook for **1.1.2 Set Declaration vs. Definition** in the AMPL lab Documentation, section "Giving Members in Data, and Changing Them with `let`". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Giving Members in Data, and Changing Them with `let`

A declared set gets its members in a data section with `:=`, or from Python. Assigning a new list
from Python replaces the members, so Python list operations can add or remove members:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;   # products; members come from data

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils", "plate"]

In [5]:
# run.py
members = ampl.set["PROD"].to_list()
members = members + ["rods"]                         # add a member
members = [p for p in members if p != "bands"]       # remove a member
ampl.set["PROD"] = members
print(ampl.set["PROD"].to_list())

['coils', 'plate', 'rods']


### Pure AMPL: data.dat and model.run

Pure AMPL. In a script, `let` replaces the members, and set
expressions can add or remove members:

In [6]:
%%writefile data.dat
set PROD := bands coils plate;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
let PROD := PROD union {"rods"};     # add a member
let PROD := PROD diff {"bands"};     # remove a member
display PROD;

set PROD := coils plate rods;

